# Databel: validación independiente

Este notebook complementa la plantilla de Power BI. Datos ficticios del caso educativo de DataCamp. Evalúa una instantánea, no una serie temporal.

In [1]:
from pathlib import Path
import pandas as pd
root = Path.cwd()
archivo = root / 'data/Databel - Data.csv'
df = pd.read_csv(archivo)
assert df['Customer ID'].is_unique
assert set(df['Churn Label']) == {'Yes', 'No'}
df.shape

(6687, 29)

## Indicadores
La tasa se calcula sobre todos los clientes del contexto. No es una tasa mensual.

In [2]:
clientes = len(df)
bajas = int(df['Churn Label'].eq('Yes').sum())
pd.Series({'Clientes': clientes, 'Bajas': bajas, 'Tasa de abandono': bajas/clientes})

Clientes            6687.000000
Bajas               1796.000000
Tasa de abandono       0.268581
dtype: float64

## Contratos
Una tasa alta y un volumen alto son señales distintas. La comparación es descriptiva.

In [3]:
resumen = df.assign(baja=df['Churn Label'].eq('Yes')).groupby('Contract Type')['baja'].agg(clientes='size', bajas='sum')
resumen['tasa'] = resumen['bajas']/resumen['clientes']
assert resumen['clientes'].sum() == clientes
assert resumen['bajas'].sum() == bajas
resumen

,clientes,bajas,tasa
Contract Type,,,
Month-to-Month,3411,1579,0.462914
One Year,1479,167,0.112914
Two Year,1797,50,0.027824


## Motivos ausentes
Se conservan 27 bajas sin motivo. El denominador de los porcentajes es el total de bajas.

In [4]:
perdidos = df.loc[df['Churn Label'].eq('Yes')]
motivos = perdidos['Churn Category'].fillna('Sin motivo informado').value_counts().to_frame('bajas')
motivos['porcentaje_de_bajas'] = motivos['bajas']/bajas
assert motivos['bajas'].sum() == bajas
motivos

,bajas,porcentaje_de_bajas
Churn Category,,
Competitor,805,0.448218
Attitude,287,0.159800
Dissatisfaction,286,0.159243
Price,200,0.111359
Other,191,0.106347
Sin motivo informado,27,0.015033


## Plan internacional
Cruzo actividad y suscripción para evitar atribuir al plan la composición de sus usuarios.

In [5]:
cruce = df.assign(baja=df['Churn Label'].eq('Yes')).groupby(['Intl Active','Intl Plan'])['baja'].agg(clientes='size', bajas='sum')
cruce['tasa'] = cruce['bajas']/cruce['clientes']
cruce

clientes  bajas      tasa
Intl Active Intl Plan                           
No          no             3939    788  0.200051
            yes             177    126  0.711864
Yes         no             2097    846  0.403433
            yes             474     36  0.075949

## Comprobación del modelo propio
La plantilla cargó en Power BI Desktop. Las consultas DAX se contrastaron con pandas en 13 contextos de filtro y para las 11 combinaciones de dimensiones y medidas de los gráficos. Esta celda resume la evidencia guardada; no vuelve a conectarse a Power BI. Las nueve medidas y Power Query coinciden con la fuente publicada.

In [6]:
import json
validacion = json.loads((root / 'output/validacion_motor.json').read_text(encoding='utf-8-sig'))
assert validacion['status'] == 'OK'
assert validacion['source_matches_loaded_model']
pd.Series({'Pruebas de filtros': len(validacion['filter_cases']), 'Combinaciones de gráficos': len(validacion['visual_cases']), 'Medidas comprobadas': validacion['measures'], 'Fallos': len(validacion['failures'])})

Pruebas de filtros           13
Combinaciones de gráficos    11
Medidas comprobadas           9
Fallos                        0
dtype: int64

## Conclusión
Los contratos mensuales concentran 1 579 bajas y una tasa del 46,29 %. La competencia representa el 44,82 % de todas las bajas. Estas asociaciones orientan una investigación, pero no prueban causalidad.

La revisión conserva 51 estados y territorios y 21 motivos, añade un filtro de contrato y contextualiza las tasas con clientes y bajas. El cálculo del modelo propio está validado. Sigue pendiente comprobar con clics los filtros, tooltips y páginas en Power BI: las consultas no sustituyen esa revisión visual. El procedimiento y sus límites están en docs/validacion_dashboard.md.